# Aurora–Solar Correlation Analysis
## Correlation between Auroral Indices, Solar Indices, and Neutron Monitor Count Rates

**Author**: Natdaporn T.  
**Project Code**: 33E-PH03-01
**Institution**: Mahidol Wittayanusorn School
**Nakhon Pathom, Thailand**

---

This notebook investigates the correlation between:
- **Auroral indices**: AE (Auroral Electrojet), Dst (Disturbance Storm Time), Kp (Planetary K-index)
- **Solar indices**: SSN (Sunspot Number), F10.7 (10.7 cm Solar Radio Flux)
- **Neutron Monitor count rates**: OULU, THUL, APTY, PSNM, JUNG1, HUAN

**Data period**: 1964–2020  
**Analysis methods**: Pearson correlation, lagged cross-correlation, hysteresis analysis across solar cycles


## 1. Setup & Imports

In [ ]:
# Core libraries
import pandas as pd
import numpy as np
import re
from datetime import datetime, timedelta
from functools import reduce
import math

# Visualization
import matplotlib.pyplot as plt
import plotly.io as pio
import plotly.express as px
import plotly.graph_objects as go
import plotly.colors as pc
from plotly.subplots import make_subplots
pio.renderers.default = "notebook"

# Machine learning
from sklearn.linear_model import LinearRegression


In [ ]:
# Verify Plotly is working
import plotly
print(f"Plotly version: {plotly.__version__}")


## 2. Utility Functions

### 2.1 String parsing helpers

In [ ]:
def sep(s):
    """Split a string by hyphens, colons, or whitespace into float parts."""
    parts = re.split(r'[-:\s]+', s.strip())
    real = [float(p) for p in parts if p]
    return real


def is_valid(s):
    """Check if a string can be parsed by sep()."""
    try:
        sep(str(s))
        return True
    except:
        return False


def safe_sep(s):
    """Safe wrapper for sep() that returns None on failure."""
    try:
        return sep(str(s))
    except:
        return None


### 2.2 Time and statistics helpers

In [ ]:
def to_fractional_year(dt):
    """Convert a datetime to fractional year (e.g., 2015.5 = July 2, 2015)."""
    if pd.isna(dt):
        return np.nan
    year_start = datetime(dt.year, 1, 1)
    next_year_start = datetime(dt.year + 1, 1, 1)
    year_length = (next_year_start - year_start).total_seconds()
    seconds_into_year = (dt - year_start).total_seconds()
    return round(dt.year + seconds_into_year / year_length, 6)


def fractional_date_from_str(date_str, time_str):
    """Convert date and time strings to fractional year."""
    dt = datetime.strptime(f"{date_str} {time_str}", "%Y-%m-%d %H:%M:%S")
    year_start = datetime(dt.year, 1, 1)
    next_year_start = datetime(dt.year + 1, 1, 1)
    year_length = (next_year_start - year_start).total_seconds()
    elapsed = (dt - year_start).total_seconds()
    return round(dt.year + elapsed / year_length, 6)


def filter_iqr(group, col, factor=1.5):
    """Replace outliers (outside factor*IQR) with NaN."""
    Q1 = group[col].quantile(0.25)
    Q3 = group[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - factor * IQR
    upper = Q3 + factor * IQR
    group = group.copy()
    mask = (group[col] < lower) | (group[col] > upper)
    group.loc[mask, col] = np.nan
    return group


## 3. Data Processing Functions

Each function reads a raw data file and returns a clean DataFrame with `datetime` and the parameter column.

### 3.1 Geomagnetic Data — Polar Field

In [ ]:
def process_geo(filepath):
    """Process WSO polar magnetic field data."""
    df = pd.read_csv(filepath, sep=r'\s+', comment='#', engine='python')
    df = df.loc[:, ~df.columns.str.contains('filt')]
    df = df.rename(columns={'min': 'minute', 'secound': 'second'})
    df = df.apply(pd.to_numeric, errors='coerce')
    df['datetime'] = pd.to_datetime(df[['year', 'month', 'day', 'hour', 'minute', 'second']])

    def datetime_to_decimal_year(dt):
        year_start = pd.Timestamp(year=dt.year, month=1, day=1)
        year_end = pd.Timestamp(year=dt.year + 1, month=1, day=1)
        return dt.year + (dt - year_start).total_seconds() / (year_end - year_start).total_seconds()

    df['Date'] = df['datetime'].apply(datetime_to_decimal_year)
    df = df.drop(columns=['year', 'month', 'day', 'hour', 'minute', 'second'])
    cols = ['Date'] + [c for c in df.columns if c != 'Date']
    return df[cols]


### 3.2 Auroral Electrojet (AE) Index

In [ ]:
def process_ae(filepath):
    """Process AE index data from WDC Kyoto IAGA-2002 format files."""
    # Try reading with different delimiters
    try:
        df = pd.read_csv(filepath)
    except:
        df = pd.read_csv(filepath, delimiter='\t')
    
    df.drop_duplicates(inplace=True)
    df.columns = df.columns.str.strip()
    df.rename(columns={'Format                 IAGA-2002                                    |': 'data'}, inplace=True)
    
    # Create columns for parsed data
    col_names = ['year', 'month', 'day', 'hour', 'minute', 'second', 'doy', 'ae', 'au', 'al', 'ao']
    df[col_names] = np.nan
    
    # Parse valid rows
    df = df[df['data'].apply(is_valid)].reset_index(drop=True)
    df['data'] = df['data'].apply(safe_sep)
    parsed = pd.DataFrame(df['data'].to_list(), columns=col_names)
    df.loc[:, col_names] = parsed.values
    
    df = df.drop(['data', 'doy'], axis=1)
    df['datetime'] = pd.to_datetime(df[['year', 'month', 'day', 'hour', 'minute', 'second']])
    df['ae'] = df['ae'].replace(0, np.nan)
    df = filter_iqr(df, 'ae')
    return df[['ae', 'datetime']]


### 3.3 Neutron Monitor Data

In [ ]:
# --- OULU Neutron Monitor (Finland) ---
def process_oulu(filepath):
    """Process OULU neutron monitor CSV data (counts/min -> counts/sec)."""
    df = pd.read_csv(filepath)
    df['datetime'] = pd.to_datetime(df['Timestamp'], errors='coerce').dt.tz_localize(None)
    df = df.rename(columns={'CorrectedCountRate[cts/min]': 'oulu'})
    df['oulu'] = df['oulu'] / 60.0  # Convert to counts/second
    df = filter_iqr(df, 'oulu')
    return df[['datetime', 'oulu']]


# --- Generic HTML Neutron Monitor parser ---
def _process_nm_html(filepath, col_name):
    """Parse neutron monitor HTML data (common format for THUL, APTY, HUAN, JUNG1)."""
    with open(filepath, 'r', encoding='utf-8') as f:
        lines = f.readlines()
    
    data = []
    pattern = r'(\d{4}\.\d{2}\.\d{2}) (\d{2}:\d{2})\s+(\d+)'
    for line in lines:
        line = line.strip()
        match = re.match(pattern, line)
        if match:
            date, time, imp_min = match.group(1), match.group(2), float(match.group(3))
        else:
            date, time, imp_min = None, None, np.nan
        data.append({'date': date, 'time': time, col_name: imp_min})
    
    df = pd.DataFrame(data)
    df['datetime'] = pd.to_datetime(df['date'] + ' ' + df['time'], format='%Y.%m.%d %H:%M', errors='coerce')
    df[col_name] = df[col_name].astype(float)
    df = filter_iqr(df, col_name)
    df[col_name] = df[col_name] // 60.0  # Convert to counts/sec
    return df[[col_name, 'datetime']]


def process_thul(filepath):
    """Process THULE neutron monitor data (Greenland)."""
    return _process_nm_html(filepath, 'thul')


def process_apty(filepath):
    """Process APATITY neutron monitor data (Russia)."""
    return _process_nm_html(filepath, 'apty')


def process_huan(filepath):
    """Process HUANCAYO neutron monitor data (Peru)."""
    df = _process_nm_html(filepath, 'apty')
    df = df.rename(columns={'apty': 'huan'})
    return df


def process_jung1(filepath):
    """Process JUNGFRAUJOCH neutron monitor data (Switzerland)."""
    df = _process_nm_html(filepath, 'apty')
    df = df.rename(columns={'apty': 'jung1'})
    return df


# --- PSNM Neutron Monitor (McMurdo, Antarctica) ---
def process_psnm(filepath):
    """Process PSNM neutron monitor data from semicolon-delimited text format."""
    with open(filepath) as f:
        lines = f.readlines()
    
    # Find header line
    for line in lines:
        if line.strip().startswith('start_date_time'):
            break
    start_index = lines.index(line) + 1
    
    data_lines = []
    for line in lines[start_index:]:
        line = line.strip()
        if not line:
            continue
        try:
            datetime_str, count = line.split(';')
            date_part, time_part = datetime_str.split()
            count = float(count)
        except:
            date_part, time_part, count = None, None, np.nan
        data_lines.append([date_part, time_part, count])
    
    df = pd.DataFrame(data_lines, columns=['date', 'time', 'psnm'])
    df['datetime'] = pd.to_datetime(df['date'] + ' ' + df['time'], errors='coerce')
    df = df[['datetime', 'psnm']]
    df = filter_iqr(df, 'psnm')
    return df


### 3.4 Solar & Geomagnetic Indices (Kp, SSN, F10.7, Dst)

In [ ]:
def process_kp(filepath):
    """Process Kp index data (3-hourly planetary geomagnetic index)."""
    df = pd.read_csv(filepath, sep=r'\s+', header=None)
    cols_to_keep = [0, 1, 2] + list(range(7, 15))
    df = df.iloc[:, cols_to_keep].copy()
    df.columns = ['Year', 'Month', 'Day', 'Kp1', 'Kp2', 'Kp3', 'Kp4', 'Kp5', 'Kp6', 'Kp7', 'Kp8']
    df['Date'] = pd.to_datetime(df[['Year', 'Month', 'Day']])
    
    # Melt 3-hourly Kp values into rows
    df_melt = df.melt(id_vars=['Date'], 
                       value_vars=['Kp1', 'Kp2', 'Kp3', 'Kp4', 'Kp5', 'Kp6', 'Kp7', 'Kp8'],
                       var_name='Kp_hour', value_name='Kp')
    hour_map = {'Kp1': 0, 'Kp2': 3, 'Kp3': 6, 'Kp4': 9, 'Kp5': 12, 'Kp6': 15, 'Kp7': 18, 'Kp8': 21}
    df_melt['Hour'] = df_melt['Kp_hour'].map(hour_map)
    df_melt['datetime'] = df_melt.apply(lambda row: row['Date'] + timedelta(hours=row['Hour']), axis=1)
    df_melt = filter_iqr(df_melt, 'Kp')
    return df_melt[['datetime', 'Kp']]


def process_ssn(filepath):
    """Process Sunspot Number (SSN) data."""
    df = pd.read_csv(filepath, sep=r'\s+', header=None)
    df = df.iloc[:, [0, 1, 2, 24]].copy()
    df.columns = ['Year', 'Month', 'Day', 'SSN']
    df['datetime'] = pd.to_datetime(df[['Year', 'Month', 'Day']])
    df = filter_iqr(df, 'SSN')
    return df[['datetime', 'SSN']]


def process_fradio(filepath):
    """Process F10.7 solar radio flux data."""
    df = pd.read_csv(filepath, sep=r'\s+', header=None)
    df = df.iloc[:, [0, 1, 2, 26]].copy()
    df.columns = ['Year', 'Month', 'Day', 'F10.7']
    df['datetime'] = pd.to_datetime(df[['Year', 'Month', 'Day']])
    df = filter_iqr(df, 'F10.7')
    return df[['datetime', 'F10.7']]


def process_dst(filepath):
    """Process Dst (Disturbance Storm Time) index data."""
    data_rows = []
    with open(filepath, 'r') as f:
        for line in f:
            line = line.strip()
            if re.match(r'^\d{4}-\d{2}-\d{2}\s+\d{2}:\d{2}:\d{2}\.\d+\s+\d+\s+[-+]?\d+\.?\d*$', line):
                parts = line.split()
                data_rows.append(parts)
    
    df = pd.DataFrame(data_rows, columns=['DATE', 'TIME', 'DOY', 'DST'])
    df['DST'] = df['DST'].astype(float)
    df['datetime'] = pd.to_datetime(df['DATE'] + ' ' + df['TIME'])
    df = df.rename(columns={'DST': 'dst'})
    df = filter_iqr(df, 'dst')
    return df[['dst', 'datetime']]


## 4. Polar Magnetic Field & Solar Cycle Definitions

In [ ]:
# Load polar magnetic field data
DATA_DIR = '../data/'
polarfield = process_geo(DATA_DIR + 'WSO_polarfield.txt')
polarfield = polarfield[(polarfield != -9999).all(axis=1)]
print(f"Polar field records: {len(polarfield)}")


In [ ]:
def get_intervals(df, x_col='Date'):
    """Find intervals where polar field sign changes (North ↔ South)."""
    df = df.copy()
    df['A'] = df['Nf'] - df['Sf']
    intervals = []
    start = df[x_col].iloc[0]
    current_sign = df['A'].iloc[0] > 0
    
    for i in range(1, len(df)):
        sign = df['Avgf'].iloc[i] > 0
        if sign != current_sign:
            intervals.append((start, df[x_col].iloc[i], current_sign))
            start = df[x_col].iloc[i]
            current_sign = sign
    intervals.append((start, df[x_col].iloc[-1], current_sign))
    return intervals


# Pre-computed polar field intervals (avoid recomputing each run)
polars = [
    (np.float64(1976.4149727092188), np.float64(1980.0843716163226), np.True_),
    (np.float64(1980.0843716163226), np.float64(1990.1037808536275), np.False_),
    (np.float64(1990.1037808536275), np.float64(2000.12535522288), np.True_),
    (np.float64(2000.12535522288), np.float64(2013.210630168696), np.False_),
    (np.float64(2013.210630168696), np.float64(2023.8626849632167), np.True_),
]


def get_poles(year):
    """Return 'North' or 'South' based on solar polar field at given year."""
    for start, end, is_north in polars:
        if start <= year <= end:
            return 'North' if is_north else 'South'
    return 'Unknown'


# Solar cycle definitions (approximate dates)
cycles = [
    (np.float64(1964.75), np.float64(1976.1667), 'Cycle 20'),
    (np.float64(1976.1667), np.float64(1986.6667), 'Cycle 21'),
    (np.float64(1986.6667), np.float64(1996.5833), 'Cycle 22'),
    (np.float64(1996.5833), np.float64(2008.9167), 'Cycle 23'),
    (np.float64(2008.9167), np.float64(2019.9167), 'Cycle 24'),
    (np.float64(2019.9167), np.float64(2025.0), 'Cycle 25'),
]


def get_cycles(year):
    """Return solar cycle label for a given year."""
    for start, end, label in cycles:
        if start <= year <= end:
            return label
    return 'Unknown'



### Parameter categories for analysis:
- **Solar indices**: SSN, F10.7  
- **Auroral indices**: AE, Dst, Kp  
- **Neutron monitors**: OULU, THUL, APTY, PSNM, JUNG1, HUAN


In [ ]:
solar_indices = ['SSN', 'F10.7']
aurora_indices = ['ae', 'dst', 'Kp']
neutrons = ['oulu', 'thul', 'apty', 'psnm', 'jung1', 'huan']


## 5. Data Loading & Merging

_All datasets are loaded, cleaned, and merged into a single hourly-resolution DataFrame._

In [ ]:
# Create hourly datetime index for 1964–2020
full_index = pd.date_range(start='1964-01-01', end='2020-01-01', freq='h')
df_full = pd.DataFrame({'datetime': full_index})
df_full['Date'] = df_full['datetime'].apply(to_fractional_year).round(6)
print(f"Expected time range: {full_index[0]} → {full_index[-1]}")
print(f"Total hours: {len(full_index):,}")


In [ ]:
# --- Load all datasets ---
D = DATA_DIR

# AE (Auroral Electrojet) Index
ae = pd.concat([
    process_ae(D + 'WWW_dstae03841155.dat.txt'),
    process_ae(D + 'WWW_dstae01284110.dat.txt'),
    process_ae(D + 'WWW_dstae02651896.dat.txt')
], ignore_index=True)

# OULU Neutron Monitor
oulu = pd.concat([
    process_oulu(D + 'OULU_1964_04_01 _00_00_1982_12_31 _23_30.csv'),
    process_oulu(D + 'OULU_1983_01_01 _00_00_2003_01_01 _00_00.csv'),
    process_oulu(D + 'OULU_2003_01_01 _00_00_2024_05_31 _00_00.csv')
], ignore_index=True)

# Dst Index
dst = pd.concat([
    process_dst(D + 'WWW_dstae03212484.dat.txt'),
    process_dst(D + 'WWW_dstae02459993.dat.txt')
], ignore_index=True)

# Neutron Monitors
apty = pd.concat([
    process_apty(D + 'apty.html'),
    process_apty(D + 'Apatity cosmic ray station2.html')
], ignore_index=True)

huan = process_huan(D + 'huan-nm.html')
jung1 = process_jung1(D + 'jung1.html')

thul = pd.concat([
    process_thul(D + 'thulefull.html'),
    process_thul(D + 'thuleadd.html')
], ignore_index=True)

# Kp, SSN, F10.7 indices
kp = pd.concat([
    process_kp(D + 'kpdata.txt'),
    process_kp(D + 'kpdata1964.txt')
], ignore_index=True)

ssn = pd.concat([
    process_ssn(D + 'kpdata1964.txt'),
    process_ssn(D + 'kpdata.txt')
], ignore_index=True)

fradio = pd.concat([
    process_fradio(D + 'kpdata.txt'),
    process_fradio(D + 'kpdata1964.txt')
], ignore_index=True)

# PSNM (McMurdo Neutron Monitor)
psnm = process_psnm(D + '03Jan01_000000_24Jun01_000000_corr_for_pressure_1h.txt')

print("All datasets loaded successfully!")


In [ ]:
# Merge all datasets on datetime (left join to preserve hourly grid)
datasets = [df_full, ae, oulu, dst, apty, psnm, thul, kp, ssn, fradio, huan, jung1]
all_data = reduce(lambda left, right: pd.merge(left, right, on='datetime', how='left'), datasets)

# Filter to 1964–2020 and clean up
all_data = all_data.loc[all_data['Date'] <= 2020].reset_index(drop=True)
all_data['ae'] = all_data['ae'].replace(99999, np.nan)
all_data['thul'] = all_data['thul'].replace(0, np.nan)
all_data['huan'] = all_data['huan'].replace(0, np.nan)

# Add polar field hemisphere and solar cycle labels
all_data['polar'] = all_data['Date'].apply(get_poles)
all_data['cycle'] = all_data['Date'].apply(get_cycles)

print(f"Merged dataset: {len(all_data):,} rows × {len(all_data.columns)} columns")
print(f"Columns: {list(all_data.columns)}")


## 6. Data Validation

In [ ]:
# Check for missing hours
print(f"Expected hours: {len(df_full):,}")
print(f"Actual rows:    {len(all_data):,}")
missing = df_full[~df_full['Date'].isin(all_data['Date'])]
print(f"Missing hours:  {len(missing)}")


In [ ]:
# Preview the merged dataset
all_data[['datetime', 'Date', 'oulu', 'Kp', 'SSN', 'ae', 'dst', 'polar', 'cycle']].tail(10)


In [ ]:
# Basic statistics for key parameters
all_data[['oulu', 'Kp', 'SSN', 'ae', 'dst', 'thul', 'apty', 'psnm', 'jung1', 'huan', 'F10.7']].describe()


## 7. Correlation & Analysis Functions

In [ ]:
def pure_pearson(df, col1, col2):
    """Compute Pearson correlation coefficient from scratch."""
    x = df[col1].values
    y = df[col2].values
    mean_x = x.mean()
    mean_y = y.mean()
    numerator = ((x - mean_x) * (y - mean_y)).sum()
    denominator = ((x - mean_x)**2).sum()**0.5 * ((y - mean_y)**2).sum()**0.5
    if denominator == 0:
        return 0
    return numerator / denominator


In [ ]:
def lagged_correlation(df, col1, col2, polar='both', lag=0, rolling=0, norm=True):
    """
    Compute Pearson correlation with optional time lag, rolling average, 
    normalization, and polar hemisphere filter.
    
    Parameters:
        df: DataFrame with time series data
        col1, col2: column names to correlate
        polar: 'North', 'South', or 'both'
        lag: shift col2 by N steps (positive = col2 leads col1)
        rolling: rolling window size for smoothing (>1 enables)
        norm: apply min-max normalization before correlation
    Returns:
        Pearson correlation coefficient (float or NaN)
    """
    df1 = df.copy()
    
    if polar == 'North':
        df1 = df1[df1['polar'] == 'North']
    elif polar == 'South':
        df1 = df1[df1['polar'] == 'South']
    
    if not set([col1, col2]).issubset(df1.columns):
        raise KeyError(f"Columns {col1}, {col2} not found")
    
    df1 = df1[[col1, col2]].copy()
    
    if lag != 0:
        df1[col2] = df1[col2].shift(lag)
    
    if rolling > 1:
        df1[col1] = df1[col1].rolling(window=rolling, center=True, min_periods=1).mean()
        df1[col2] = df1[col2].rolling(window=rolling, center=True, min_periods=1).mean()
    
    df1 = df1.dropna()
    if df1.empty:
        return np.nan
    
    if norm:
        def min_max_norm(series):
            return (series - series.min()) / (series.max() - series.min())
        df1[col1] = min_max_norm(df1[col1])
        df1[col2] = min_max_norm(df1[col2])
    
    return pure_pearson(df1, col1, col2)


In [ ]:
def block_average(series, window, df):
    """Compute block (non-overlapping) averages with year labels."""
    n = len(series)
    averaged = []
    years = []
    for i in range(0, n, window):
        chunk = series.iloc[i:i + window]
        averaged.append(chunk.mean())
        mid_index = int(i + (len(chunk) - 1) / 2)
        year_fraction = df.loc[mid_index, 'Date']
        year_int = int(year_fraction)
        years.append(year_int)
    return pd.Series(averaged, index=years)


## 8. Visualization Functions

In [ ]:
def plot_normed_rolling(df, x_col, y_cols, y_name='', rolling=1, norm=True, 
                         lag=0, line=1, title='', x_name='Year', save=True):
    """
    Plot time series with optional normalization and rolling average.
    
    Parameters:
        df: DataFrame
        x_col: column for x-axis ('Date' for time)
        y_cols: list of columns to plot on y-axis
        rolling: window size for rolling average (1 = no smoothing)
        norm: apply min-max normalization
        lag: shift second y column by N steps
        save: if True, save to results/ folder
    """
    from datetime import datetime as dt_now
    
    df1 = df.copy()
    
    def min_max_norm(series):
        return (series - series.min()) / (series.max() - series.min())
    
    if lag != 0 and len(y_cols) == 2 and x_col == 'Date':
        df1[y_cols[1]] = df1[y_cols[1]].shift(lag)
    
    subset = y_cols + ([x_col] if x_col in df1.columns else [])
    df1 = df1.dropna(subset=subset).copy()
    
    if rolling > 1:
        for col in y_cols:
            df1[col] = df1[col].rolling(window=rolling, center=True, min_periods=1).mean()
    
    if norm:
        for col in y_cols:
            df1[col] = min_max_norm(df1[col])
    
    if len(y_cols) == 2:
        corr = lagged_correlation(df, y_cols[0], y_cols[1], lag=lag, rolling=rolling, norm=norm)
        print(f'Correlation ({y_cols[0]} vs {y_cols[1]}): {corr:.4f}')
    
    fig = go.Figure()
    colors = ['black', 'blue', 'green', 'red', 'purple', 'orange', 'brown']
    
    for i, col in enumerate(y_cols):
        slope, intercept = np.polyfit(df1[x_col], df1[col], 1)
        trend_y = intercept + slope * df1[x_col]
        
        fig.add_trace(go.Scatter(
            x=df1[x_col], y=df1[col],
            mode='lines' if line == 1 else 'markers',
            name=col.upper(),
            marker=dict(size=0),
            line=dict(color=colors[i % len(colors)])
        ))
        
        if x_col != 'Date':
            fig.add_trace(go.Scatter(
                x=df1[x_col], y=trend_y,
                mode='lines',
                name=f'Trend (slope={slope:.3f})',
                line=dict(color='red', dash='dash')
            ))
    
    fig.update_layout(
        title={'text': title, 'x': 0.5, 'xanchor': 'center'},
        title_font=dict(size=26, family='Courier New', color='black'),
        plot_bgcolor='white', paper_bgcolor='white',
        font=dict(family='Courier New', size=22, color='black'),
        legend=dict(font=dict(size=18, family='Courier New', color='black')),
        xaxis_title=x_name, yaxis_title=y_name
    )
    
    if save:
        now = dt_now().strftime('%Y-%m-%d_%H-%M-%S')
        filename = f'../results/plot_{now}.png'
        fig.write_image(filename)
        print(f'Saved: {filename}')
    
    fig.show()


## 9. Correlation Matrix Functions

In [ ]:
def corr_matrix(df, cols, polar='both', lag=0, save=True):
    """
    Compute and plot a correlation matrix for selected columns.
    
    Parameters:
        df: DataFrame
        cols: list of column names to include
        polar: 'North', 'South', or 'both'
        lag: time lag for correlation
        save: save plot to results/
    """
    matrix = pd.DataFrame(index=cols, columns=cols, dtype=float)
    for i in cols:
        for j in cols:
            corr = lagged_correlation(df, i, j, polar=polar, lag=lag, rolling=0, norm=True)
            matrix.loc[i, j] = corr
    
    matrix = matrix.astype(float)
    
    # Rename for display
    rename_map = {
        'oulu': 'OULU NM', 'apty': 'APTY NM', 'psnm': 'PSNM NM', 'thul': 'THUL NM',
        'ae': 'AE Index', 'Kp': 'Kp Index', 'dst': 'DST Index'
    }
    matrix_display = matrix.rename(index=rename_map, columns=rename_map)
    
    fig = go.Figure(data=go.Heatmap(
        z=matrix_display.values,
        x=matrix_display.columns,
        y=matrix_display.index,
        colorscale='RdBu_r',
        zmid=0,
        text=matrix_display.round(3).values,
        texttemplate='%{text}',
        textfont={'size': 14}
    ))
    
    fig.update_layout(
        title=f'Correlation Matrix (polar={polar}, lag={lag})',
        width=800, height=700
    )
    
    if save:
        from datetime import datetime as dt_now
        now = dt_now().strftime('%Y-%m-%d_%H-%M-%S')
        fig.write_image(f'../results/corr_mat_plot_{now}.svg')
    
    fig.show()
    return matrix


## 10. Lagged Correlation Analysis

In [ ]:
def corr_lag(norm=True, save=True):
    """Plot correlation vs time lag for neutron monitors vs solar/auroral indices."""
    rangelag = range(-400, 101, 1)
    df = all_data.copy()
    
    xlist = list(rangelag)
    ylist_oulu_ssn = [lagged_correlation(df, 'SSN', 'oulu', lag=lag, rolling=0, norm=norm) for lag in rangelag]
    ylist_oulu_kp = [lagged_correlation(df, 'Kp', 'oulu', lag=lag, rolling=0, norm=norm) for lag in rangelag]
    ylist_oulu_ae = [lagged_correlation(df, 'ae', 'oulu', lag=lag, rolling=0, norm=norm) for lag in rangelag]
    
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=xlist, y=ylist_oulu_ssn, mode='lines', name='OULU vs SSN'))
    fig.add_trace(go.Scatter(x=xlist, y=ylist_oulu_kp, mode='lines', name='OULU vs Kp'))
    fig.add_trace(go.Scatter(x=xlist, y=ylist_oulu_ae, mode='lines', name='OULU vs AE'))
    
    fig.update_layout(
        title='Lagged Correlation: OULU Neutron Monitor vs Indices',
        xaxis_title='Lag (hours)',
        yaxis_title='Pearson Correlation',
        plot_bgcolor='white'
    )
    
    if save:
        from datetime import datetime as dt_now
        now = dt_now().strftime('%Y-%m-%d_%H-%M-%S')
        fig.write_image(f'../results/lag_corr_plot_{now}.png')
    
    fig.show()


## 11. Solar Cycle Hysteresis Analysis

In [ ]:
def ssn_hysteresis_custom(cycles_list, y_axis='SSN', window_size=24*30*3):
    """
    Plot hysteresis loops: neutron monitor count rate vs solar index 
    for selected solar cycles.
    
    Parameters:
        cycles_list: list of cycle numbers to plot [21, 22, 23, 24]
        y_axis: solar index for y-axis ('SSN' or 'F10.7')
        window_size: block averaging window in hours
    """
    df = all_data.copy()
    colors = ['red', 'blue', 'green']
    nm_stations = ['oulu', 'thul', 'jung1']
    
    n_cycles = len(cycles_list)
    cols = min(2, n_cycles)
    rows = (n_cycles + 1) // 2
    
    fig = make_subplots(
        rows=rows, cols=cols,
        subplot_titles=[f'Cycle {c}' for c in cycles_list]
    )
    
    for idx, cycle_num in enumerate(cycles_list):
        row = idx // 2 + 1
        col = idx % 2 + 1
        
        df_cycle = df[df['cycle'] == f'Cycle {cycle_num}'].reset_index(drop=True)
        
        for j, nm in enumerate(nm_stations):
            if nm not in df_cycle.columns:
                continue
            nm_avg = block_average(df_cycle[nm], window_size, df_cycle)
            ssn_avg = block_average(df_cycle[y_axis], window_size, df_cycle)
            
            common_idx = nm_avg.index.intersection(ssn_avg.index)
            
            fig.add_trace(go.Scatter(
                x=nm_avg[common_idx].values,
                y=ssn_avg[common_idx].values,
                mode='lines+markers',
                name=nm.upper(),
                line=dict(color=colors[j % len(colors)]),
                marker=dict(size=5)
            ), row=row, col=col)
    
    fig.update_layout(
        title=f'Hysteresis: Neutron Monitors vs {y_axis}',
        height=300 * rows, width=600 * cols,
        showlegend=True
    )
    
    # Save
    from datetime import datetime as dt_now
    now = dt_now().strftime('%Y-%m-%d_%H-%M-%S')
    fig.write_image(f'../results/hysteresis_custom_{now}_{y_axis}.png')
    fig.show()
    
    return fig


## 12. Results & Visualizations

Run the cells below to generate analysis plots. Outputs are saved to the `results/` folder.

### 12.1 Time Series Overview

In [ ]:
# Plot Kp Index over time
plot_normed_rolling(all_data, 'Date', ['Kp'], 'Kp Index', 
                     rolling=150, norm=False, title='Kp Index (1964–2020)', save=True)


In [ ]:
# Compare SSN, Kp, AE, and OULU neutron count (normalized)
plot_normed_rolling(all_data, 'Date', ['SSN', 'Kp', 'ae', 'oulu'], '',
                     rolling=150, norm=True, 
                     title='Normalized Solar, Auroral & Neutron Monitor Indices', save=True)


### 12.2 Correlation Between Kp Index and Neutron Count Rate

In [ ]:
plot_normed_rolling(all_data, 'oulu', ['Kp'], 'Kp Index',
                     rolling=24*365, norm=False,
                     title='Kp Index vs OULU Neutron Count Rate',
                     x_name='OULU NM (counts/second)', save=True)


### 12.3 Correlation Matrix

In [ ]:
# Compute correlation matrix for all parameters
matrix = corr_matrix(all_data, ['oulu', 'thul', 'apty', 'psnm', 'ae', 'Kp', 'dst', 'SSN'],
                      polar='both', lag=0, save=True)
matrix


### 12.4 Lagged Cross-Correlation

In [ ]:
# Plot correlation vs time lag
corr_lag(norm=True, save=True)


### 12.5 Solar Cycle Hysteresis

In [ ]:
# Hysteresis loops for cycles 21–24
ssn_hysteresis_custom([21, 22, 23, 24], y_axis='SSN', window_size=24*30*3)


In [ ]:
# Single cycle detail: Cycle 21
ssn_hysteresis_custom([21], y_axis='SSN', window_size=24*30*3)
